# AI Music OS — Kokoro TTS V7

Free local models; no API key. Choose **T4 GPU**, then **Run all** in a fresh Colab session. Open the printed Gradio link. Each notebook uses its own pinned Python 3.11 environment, leaving Colab's Python 3.13 packages untouched. Models and WAV outputs remain on Drive. Keep the notebook session open; free Colab can disconnect it.

Upload this notebook directly to Colab, or replace its matching file in GitHub. No other repository files are required. Dependency and CPU import tests passed locally; Colab T4 inference is not certified.


In [ ]:
MODULE = 'kokoro'
# Separate Python 3.11 environment; do not install AI packages into Colab's kernel.
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, time, re
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/AI_Music_OS')
CACHE = ROOT / 'cache' / 'huggingface'
OUT = ROOT / 'outputs' / {'ace': 'ace_step', 'kokoro': 'kokoro', 'f5': 'qwen_voice'}[MODULE]
VOICE_BANK = ROOT / 'voice_bank'
MODELS = ROOT / 'models' / 'ace_step'
for p in (CACHE, OUT, VOICE_BANK, MODELS): p.mkdir(parents=True, exist_ok=True)
ENV = os.environ.copy()
ENV['HF_HOME'] = str(CACHE)
ENV['HUGGINGFACE_HUB_CACHE'] = str(CACHE / 'hub')
ENV.pop('TRANSFORMERS_CACHE', None)
ENV['PYTHONUNBUFFERED'] = '1'
WORK = Path('/content/ai_music_v5') / MODULE
WORK.mkdir(parents=True, exist_ok=True)
LOG = OUT / 'install_v5.log'
def run_logged(cmd, cwd=None):
    with LOG.open('a') as log:
        result = subprocess.run(cmd, cwd=cwd, env=ENV, stdout=log, stderr=subprocess.STDOUT)
    if result.returncode:
        print(LOG.read_text(errors='replace')[-18000:])
        raise RuntimeError(f'Command failed ({result.returncode}). Complete output: {LOG}')
UV_HOME = Path('/content/ai_music_v5/bootstrap')
UV = UV_HOME / 'bin' / 'uv'
if not UV.exists():
    run_logged([sys.executable, '-m', 'venv', str(UV_HOME)])
    run_logged([str(UV_HOME / 'bin' / 'python'), '-m', 'pip', 'install', '--no-cache-dir', 'uv==0.13.8'])
run_logged([str(UV), 'python', 'install', '3.11.18'])
VENV = WORK / 'venv'
PY = VENV / 'bin' / 'python'
if not PY.exists(): run_logged([str(UV), 'venv', '--python', '3.11.18', str(VENV)])
print('Colab kernel:', sys.version.split()[0], '| AI runtime: Python 3.11.18')
print('Outputs:', OUT, '| Installation log:', LOG)


In [ ]:
LOCK_CONTENT = 'addict==2.4.0\naiofiles==23.2.1\nannotated-doc==0.0.6\nannotated-types==0.8.0\nanyio==4.16.0\nattrs==26.1.0\naudioread==3.1.0\nbabel==2.18.0\nblis==1.3.4\ncatalogue==2.0.10\ncertifi==2026.10.23\ncffi==2.1.1\ncharset-normalizer==3.5.2\nclick==8.5.1\ncloudpathlib==0.26.0\ncloudpickle==3.1.2\nconfection==0.1.5\ncsvw==4.2.0\ncurated-tokenizers==0.0.10\ncurated-transformers==0.1.1\ncymem==2.0.16\ndecorator==5.3.1\ndlinfo==2.0.0\ndocopt==0.6.2\nen-core-web-sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl\nespeakng-loader==0.2.4\nfastapi==0.162.7\nffmpy==1.0.0\nfilelock==4.2.10\nfsspec==2026.9.0\ngradio==5.22.0\ngradio-client==1.8.0\ngroovy==0.1.2\nh11==0.16.0\nhttpcore==1.0.9\nhttpx==0.28.1\nhuggingface-hub==0.30.2\nidna==3.20\nisodate==0.7.2\njinja2==3.1.6\njoblib==1.6.0\njsonschema==4.26.0\njsonschema-specifications==2025.9.1\nkokoro==0.9.4\nlangcodes==3.5.1\nlanguage-tags==1.3.2\nlazy-loader==0.6\nlibrosa==0.11.0\nllvmlite==0.50.0\nloguru==0.8.0\nmarkdown-it-py==4.3.0\nmarkupsafe==3.0.4\nmdurl==0.1.2\nmisaki==0.9.4\nmpmath==1.3.0\nmsgpack==1.2.4\nmurmurhash==1.0.16\nnarwhals==2.28.0\nnetworkx==3.6.1\nnum2words==0.5.14\nnumba==0.68.1\nnumpy==1.26.4\nnvidia-cublas-cu12==12.4.5.8\nnvidia-cuda-cupti-cu12==12.4.127\nnvidia-cuda-nvrtc-cu12==12.4.127\nnvidia-cuda-runtime-cu12==12.4.127\nnvidia-cudnn-cu12==9.1.0.70\nnvidia-cufft-cu12==11.2.1.3\nnvidia-curand-cu12==10.3.5.147\nnvidia-cusolver-cu12==11.6.1.9\nnvidia-cusparse-cu12==12.3.1.170\nnvidia-cusparselt-cu12==0.6.2\nnvidia-nccl-cu12==2.21.5\nnvidia-nvjitlink-cu12==12.4.127\nnvidia-nvtx-cu12==12.4.127\nopentelemetry-api==1.45.2\norjson==3.13.0\npackaging==26.3\npandas==2.3.3\nphonemizer-fork==3.3.2\npillow==11.3.0\npip==25.1.1\nplatformdirs==4.14.5\npooch==1.9.0\npreshed==3.0.15\npycparser==3.12\npydantic==2.10.6\npydantic-core==2.27.2\npydub==0.25.1\npygments==2.21.0\npyparsing==3.3.3\npython-dateutil==4.0.0\npython-multipart==0.0.33\npytz==2026.6\npyyaml==6.0.3\nrdflib==7.6.0\nreferencing==0.37.0\nregex==2026.11.9\nrequests==2.34.2\nrfc3986==1.5.0\nrich==15.0.0\nrpds-py==2026.9.1\nruff==0.17.5\nsafehttpx==0.1.7\nsafetensors==0.8.0\nscikit-learn==1.9.1\nscipy==1.13.1\nsegments==2.4.0\nsemantic-version==2.10.0\nsetuptools==84.2.1\nshellingham==1.5.4\nsmart-open==7.7.1\nsoundfile==0.13.1\nsoxr==1.1.0\nspacy==3.8.4\nspacy-curated-transformers==0.3.1\nspacy-legacy==3.0.12\nspacy-loggers==1.0.5\nsrsly==2.5.6\nstarlette==0.52.1\nsympy==1.13.1\ntermcolor==3.3.0\nthinc==8.3.11\nthreadpoolctl==3.8.0\ntokenizers==0.21.4\ntomlkit==0.13.3\ntorch==2.6.0+cu124\ntorchaudio==2.6.0+cu124\ntorchvision==0.21.0+cu124\ntqdm==4.70.1\ntransformers==4.50.0\ntriton==3.2.0\ntyper==0.29.1\ntyper-slim==0.24.0\ntyping-extensions==4.16.0\ntyping-inspection==0.4.5\ntzdata==2026.6\nuritemplate==4.2.0\nurllib3==2.8.0\nuvicorn==0.54.0\nwasabi==1.1.3\nweasel==0.4.3\nwebsockets==15.0.1\nwrapt==2.5.2\n'
CHECK_IMPORTS = 'import torch, torchaudio, torchvision, gradio, addict, misaki.en; from transformers import AlbertModel; from kokoro import KPipeline; print("Kokoro imports: OK")'
# Full dependency resolution from a pinned lock, including addict and phonemizers.
LOCK = WORK / 'requirements.lock'
LOCK.write_text(LOCK_CONTENT)
run_logged([str(UV), 'pip', 'sync', '--python', str(PY), '--no-cache',
            '--index', 'https://download.pytorch.org/whl/cu124',
            '--index', 'https://pypi.org/simple', '--index-strategy', 'unsafe-best-match', str(LOCK)])
run_logged([str(UV), 'pip', 'check', '--python', str(PY)])
run_logged([str(PY), '-c', CHECK_IMPORTS])
print('Complete installation + imports: OK')


In [ ]:
# Verify the actual AI runtime, not Colab's preinstalled torch.
run_logged([str(PY), '-c', "import torch; print('Torch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'); assert torch.cuda.is_available(), 'Select T4 GPU in Runtime > Change runtime type, then rerun.'"])
print('T4/CUDA check: OK')


In [ ]:
# Write the genuine model application; no GitHub code download needed.
APP = WORK / 'app.py'
APP.write_text("from pathlib import Path\nimport os\nOUT = Path(os.environ['AI_MUSIC_OUT'])\nfrom kokoro import KPipeline\npipelines = {'a': KPipeline(lang_code='a'), 'b': KPipeline(lang_code='b', model=False)}\npipelines['b'].model = pipelines['a'].model\n# STEP 6 — Kokoro audio generation\n\nimport numpy as np\nimport soundfile as sf\nimport time\n\nVOICE_CHOICES = [\n    'af_heart',\n    'af_bella',\n    'af_nicole',\n    'af_sarah',\n    'af_sky',\n    'am_adam',\n    'am_michael',\n    'bf_emma',\n    'bf_isabella',\n    'bm_george',\n    'bm_lewis'\n]\n\nSAMPLE_RATE = 24000\n\ndef generate_audio(text, voice='af_heart'):\n    text = (text or '').strip()\n\n    if not text:\n        raise ValueError('Text is empty.')\n\n    if voice not in VOICE_CHOICES:\n        raise ValueError(f'Unsupported voice: {voice}')\n\n    chunks = []\n\n    print('Generating with Kokoro...')\n    print('Voice:', voice)\n\n    selected = pipelines[voice[0]]\n    for item in selected(text, voice=voice):\n        if isinstance(item, tuple) and len(item) >= 3:\n            audio = item[2]\n        elif isinstance(item, dict):\n            audio = item.get('audio')\n        else:\n            audio = getattr(item, 'audio', None)\n\n        if audio is None:\n            raise RuntimeError(\n                f'Unsupported Kokoro output type: {type(item)}'\n            )\n\n        if hasattr(audio, 'detach'): audio = audio.detach().cpu().numpy()\n        array = np.asarray(audio, dtype=np.float32).reshape(-1)\n        if array.size == 0 or not np.isfinite(array).all(): raise RuntimeError('Invalid Kokoro audio.')\n        chunks.append(array)\n\n    if not chunks:\n        raise RuntimeError('Kokoro returned no audio.')\n\n    audio = np.concatenate(chunks)\n\n    filename = f'kokoro_{time.time_ns()}.wav'\n    path = OUT / filename\n\n    sf.write(path, audio, SAMPLE_RATE)\n\n    print('Audio saved:', path)\n    print('Duration:', round(len(audio) / SAMPLE_RATE, 2), 'seconds')\n\n    return str(path)\n\nprint('Generation function: OK')\n\n# STEP 7 — Smoke test\n\ntest_file = generate_audio(\n    'Hello. This is a test of the AI Music OS Kokoro voice generator.',\n    'af_heart'\n)\n\nprint('=====================================')\nprint('KOKORO TEST PASSED')\nprint('File:', test_file)\nprint('=====================================')\n\n# STEP 8 — Gradio UI\n\nimport gradio as gr\n\ndef ui_generate(text, voice):\n    try:\n        return generate_audio(text, voice)\n    except Exception as e:\n        raise gr.Error(str(e))\n\ndemo = gr.Interface(\n    fn=ui_generate,\n    inputs=[\n        gr.Textbox(\n            lines=8,\n            label='Text',\n            value='Hello. This is the AI Music OS Kokoro voice generator.'\n        ),\n        gr.Dropdown(\n            choices=VOICE_CHOICES,\n            value='af_heart',\n            label='Voice'\n        )\n    ],\n    outputs=gr.Audio(\n        label='Generated audio',\n        type='filepath'\n    ),\n    title='AI Music OS — Kokoro TTS V7',\n    description='Kokoro text-to-speech with Google Drive persistence.'\n)\n\nprint('Launching Gradio...')\ndemo.launch(share=True, server_name='0.0.0.0', server_port=7866, allowed_paths=[str(OUT)])\n")
ENV['AI_MUSIC_OUT'] = str(OUT)
ENV['AI_MUSIC_VOICE_BANK'] = str(VOICE_BANK)
COMMAND = [str(PY), str(APP)]


In [ ]:
# Launch in the isolated environment; rerunning safely replaces the previous process.
if 'proc' in globals() and proc.poll() is None:
    proc.terminate()
    try: proc.wait(timeout=20)
    except subprocess.TimeoutExpired: proc.kill(); proc.wait()
LAUNCH_LOG = OUT / 'launch_v5.log'
with LAUNCH_LOG.open('w') as log:
    proc = subprocess.Popen(COMMAND, cwd=str(OUT), env=ENV, stdout=log, stderr=subprocess.STDOUT)
url = None
for _ in range(900):
    time.sleep(2)
    text = LAUNCH_LOG.read_text(errors='replace')
    if proc.poll() is not None:
        print(text[-18000:])
        raise RuntimeError(f'Application exited. Full log: {LAUNCH_LOG}')
    match = re.search(r'https://[a-zA-Z0-9-]+\.gradio\.live', text)
    if match: url = match.group(0); break
if url is None:
    print(LAUNCH_LOG.read_text(errors='replace')[-6000:])
    raise TimeoutError(f'No share URL after 30 minutes; inspect {LAUNCH_LOG}. Rerun this cell to retry.')
print('Open Music OS:', url)
print('Keep this Colab session open. Logs:', LAUNCH_LOG)


In [ ]:
# Inspect the application log at any time.
print(LAUNCH_LOG.read_text(errors='replace')[-15000:])
